# CSE476: Agentic AI and Intelligent Automation
## Academic Task 2 (Project 2) — Group 2: Multi-Agent Loan Underwriting System

**Student Project Submission**  
- **Architecture**: Manager-Worker Delegation  
- **Framework**: Semantic Kernel  
- **Core Pattern**: Shared In-Memory Blackboard / Record  
- **Key Invariants**:
  1. Deterministic reducing-balance EMI tool (not model-hallucinated math).
  2. Stated income rule (EMI <= 40% of net monthly income).
  3. Strict governance: halts and refers immediately when income is missing (never assumes).
  4. Group-of-3 Add-on: Counter-offer specialist worker proposing maximum viable loan amount and extended tenure.


In [1]:
import os
import sys
import json
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root resolved to: {project_root}")


Project root resolved to: f:\Agenetic-AI


### 1. Initialize Semantic Kernel & Underwriting Manager
The Underwriting Manager coordinates 5 specialist workers (Document, EMI, Credit-History, Affordability, and Counter-Offer) using a shared in-memory blackboard record.

In [2]:
from agents.underwriting_manager import UnderwritingManager
from models.schemas import DecisionEnum, LoanApplication

bureau_file = project_root / "data" / "bureau_records.json"
run_log_file = project_root / "run_log.md"

manager = UnderwritingManager(
    annual_interest_rate=10.5,
    bureau_data_path=str(bureau_file),
    run_log_path=str(run_log_file)
)
print("UnderwritingManager initialized with registered Semantic Kernel plugins.")


UnderwritingManager initialized with registered Semantic Kernel plugins.


### 2. Multi-Agent Trace Visualizer Helper

In [3]:
def display_underwriting_trace(record):
    print("=" * 80)
    print(f"APPLICANT: {record.application.applicant_name} (ID: {record.application.applicant_id})")
    print("=" * 80)
    print(f"• Monthly Income   : ${record.extracted_monthly_income:,.2f}" if record.extracted_monthly_income else "• Monthly Income   : [MISSING / NOT PROVIDED]")
    print(f"• Existing EMIs    : ${record.extracted_existing_emis:,.2f}")
    print(f"• Requested Amount : ${record.extracted_requested_amount:,.2f}" if record.extracted_requested_amount else "• Requested Amount : N/A")
    print(f"• Tenure           : {record.extracted_tenure_months} months" if record.extracted_tenure_months else "• Tenure           : N/A")
    print("-" * 80)
    print("MULTI-AGENT EXECUTION TRACE (Step-by-Step):")
    for idx, trace in enumerate(record.agent_trace, 1):
        status_icon = "[PASS]" if trace.status in ["SUCCESS", "PASS"] else ("[INIT]" if trace.status == "INITIALIZED" else "[FAIL]")
        print(f"  Step {idx:02d} | {status_icon} | {trace.agent_name} | Action: {trace.action}")
        print(f"          |_ Tool Invoked: {trace.tool_used}")
    
    print("-" * 80)
    dec_val = record.decision.value if record.decision else 'PENDING'
    badge = '🟢 SANCTIONED' if dec_val == 'SANCTION' else ('🔴 DECLINED' if dec_val == 'DECLINE' else '🟡 REFERRED')
    print(f"FINAL VERDICT: {badge}")
    if record.computed_monthly_emi:
        print(f"• Reducing-Balance Monthly EMI : ${record.computed_monthly_emi:,.2f}")
    if record.emi_to_income_ratio is not None:
        print(f"• EMI-to-Income Ratio          : {record.emi_to_income_ratio:.1%} (Rule: <= 40%)")
    if record.bureau_profile:
        print(f"• Credit Bureau Score / Tier   : CIBIL {record.bureau_profile.cibil_score} ({record.credit_risk_level})")
    
    if record.counter_offer and record.counter_offer.eligible_for_counter_offer:
        print("\n[COUNTER-OFFER WORKER PROPOSAL]:")
        print(f"  1. Max Affordable Amount at Current Tenure : ${record.counter_offer.max_eligible_loan_amount:,.2f} (EMI: ${record.counter_offer.max_amount_emi:,.2f})")
        if record.counter_offer.extended_tenure_months:
            print(f"  2. Extended Tenure Option                  : {record.counter_offer.extended_tenure_months} months (New EMI: ${record.counter_offer.extended_tenure_emi:,.2f})")
        print(f"  Guidance: {record.counter_offer.recommendation_note}")
        
    print("\nONE-PARAGRAPH UNDERWRITING JUSTIFICATION:")
    print(f'> "{record.one_paragraph_justification}"\n')


### Scenario 1: Prime Applicant (Passes All Criteria -> SANCTION)
- Income: $125,000 / month
- Requested: $500,000 for 36 months
- CIBIL Score: 785 (0 defaults)
- Resulting EMI: $16,251.22 (13.0% of income, well below 40% ceiling)

In [4]:
app1 = {
    "applicant_id": "APP-101",
    "applicant_name": "Alice Sharma",
    "monthly_income": 125000.0,
    "existing_emis": 8000.0,
    "requested_amount": 500000.0,
    "tenure_months": 36,
    "purpose": "Home renovation and interior design"
}
record1 = manager.evaluate_application(app1)
display_underwriting_trace(record1)


APPLICANT: Alice Sharma (ID: APP-101)
• Monthly Income   : $125,000.00
• Existing EMIs    : $8,000.00
• Requested Amount : $500,000.00
• Tenure           : 36 months
--------------------------------------------------------------------------------
MULTI-AGENT EXECUTION TRACE (Step-by-Step):
  Step 01 | [INIT] | UnderwritingManager | Action: initiate_underwriting
          |_ Tool Invoked: manager_worker_orchestrator
  Step 02 | [PASS] | DocumentAgent | Action: validate_application_completeness
          |_ Tool Invoked: DocumentTools.validate_application_completeness
  Step 03 | [PASS] | DocumentAgent | Action: extract_financial_fields
          |_ Tool Invoked: DocumentTools.extract_financial_fields
  Step 04 | [PASS] | EMIComputationAgent | Action: compute_reducing_balance_emi
          |_ Tool Invoked: EMITools.compute_reducing_balance_emi
  Step 05 | [PASS] | EMIComputationAgent | Action: generate_amortization_schedule
          |_ Tool Invoked: EMITools.generate_amortization_schedu

### Scenario 2: Unaffordable EMI (Fails 40% Rule -> DECLINE + COUNTER-OFFER)
- Income: $35,000 / month (Max allowed EMI = $14,000)
- Requested: $1,200,000 for 24 months
- Resulting EMI: $55,651.25 (> 150% of income)
- Counter-Offer Worker activates to compute maximum viable principal and tenure options.

In [5]:
app2 = {
    "applicant_id": "APP-104",
    "applicant_name": "Bob Mehta",
    "monthly_income": 35000.0,
    "existing_emis": 10000.0,
    "requested_amount": 1200000.0,
    "tenure_months": 24,
    "purpose": "Commercial vehicle purchase"
}
record2 = manager.evaluate_application(app2)
display_underwriting_trace(record2)


APPLICANT: Bob Mehta (ID: APP-104)
• Monthly Income   : $35,000.00
• Existing EMIs    : $10,000.00
• Requested Amount : $1,200,000.00
• Tenure           : 24 months
--------------------------------------------------------------------------------
MULTI-AGENT EXECUTION TRACE (Step-by-Step):
  Step 01 | [INIT] | UnderwritingManager | Action: initiate_underwriting
          |_ Tool Invoked: manager_worker_orchestrator
  Step 02 | [PASS] | DocumentAgent | Action: validate_application_completeness
          |_ Tool Invoked: DocumentTools.validate_application_completeness
  Step 03 | [PASS] | DocumentAgent | Action: extract_financial_fields
          |_ Tool Invoked: DocumentTools.extract_financial_fields
  Step 04 | [PASS] | EMIComputationAgent | Action: compute_reducing_balance_emi
          |_ Tool Invoked: EMITools.compute_reducing_balance_emi
  Step 05 | [PASS] | EMIComputationAgent | Action: generate_amortization_schedule
          |_ Tool Invoked: EMITools.generate_amortization_schedul

### Scenario 3: Missing Income Field (Pipeline Halts Immediately -> REFER)
- Income: Missing (`None`)
- The Underwriting Manager halts at the Document Agent rather than assuming or hallucinating an income figure.
- Downstream calculation and bureau scoring are prevented.

In [6]:
app3 = {
    "applicant_id": "APP-103",
    "applicant_name": "Charlie Verma",
    "monthly_income": None,
    "existing_emis": 3000.0,
    "requested_amount": 300000.0,
    "tenure_months": 24,
    "purpose": "Personal medical emergency"
}
record3 = manager.evaluate_application(app3)
display_underwriting_trace(record3)


APPLICANT: Charlie Verma (ID: APP-103)
• Monthly Income   : [MISSING / NOT PROVIDED]
• Existing EMIs    : $0.00
• Requested Amount : N/A
• Tenure           : N/A
--------------------------------------------------------------------------------
MULTI-AGENT EXECUTION TRACE (Step-by-Step):
  Step 01 | [INIT] | UnderwritingManager | Action: initiate_underwriting
          |_ Tool Invoked: manager_worker_orchestrator
  Step 02 | [FAIL] | DocumentAgent | Action: validate_application_completeness
          |_ Tool Invoked: DocumentTools.validate_application_completeness
--------------------------------------------------------------------------------
FINAL VERDICT: 🟡 REFERRED

ONE-PARAGRAPH UNDERWRITING JUSTIFICATION:
> "The loan application for Charlie Verma (Applicant ID: APP-103) has been REFERRED for manual underwriting review. In accordance with credit policy, the automated pipeline halted immediately at the Document Verification stage because verified monthly income was absent from the ap

### Scenario 4: High Risk Credit Bureau Red Flags (DECLINE)
- Income: $95,000 / month (Affordable loan)
- CIBIL Score: 590, 2 write-offs / defaults recorded.
- Fails on credit risk despite positive cash flow.

In [7]:
app4 = {
    "applicant_id": "APP-102",
    "applicant_name": "David Khan",
    "monthly_income": 95000.0,
    "existing_emis": 5000.0,
    "requested_amount": 200000.0,
    "tenure_months": 24,
    "purpose": "Debt consolidation"
}
record4 = manager.evaluate_application(app4)
display_underwriting_trace(record4)


APPLICANT: David Khan (ID: APP-102)
• Monthly Income   : $95,000.00
• Existing EMIs    : $5,000.00
• Requested Amount : $200,000.00
• Tenure           : 24 months
--------------------------------------------------------------------------------
MULTI-AGENT EXECUTION TRACE (Step-by-Step):
  Step 01 | [INIT] | UnderwritingManager | Action: initiate_underwriting
          |_ Tool Invoked: manager_worker_orchestrator
  Step 02 | [PASS] | DocumentAgent | Action: validate_application_completeness
          |_ Tool Invoked: DocumentTools.validate_application_completeness
  Step 03 | [PASS] | DocumentAgent | Action: extract_financial_fields
          |_ Tool Invoked: DocumentTools.extract_financial_fields
  Step 04 | [PASS] | EMIComputationAgent | Action: compute_reducing_balance_emi
          |_ Tool Invoked: EMITools.compute_reducing_balance_emi
  Step 05 | [PASS] | EMIComputationAgent | Action: generate_amortization_schedule
          |_ Tool Invoked: EMITools.generate_amortization_schedule


### 3. Summary Scorecard Across All Scenarios

In [8]:
print("=" * 105)
print(f"{'Applicant':<16} | {'Req Loan':<12} | {'Income':<12} | {'Monthly EMI':<13} | {'EMI/Inc':<8} | {'CIBIL':<6} | {'Decision':<10}")
print("=" * 105)
for r in [record1, record2, record3, record4]:
    app_n = r.application.applicant_name
    req = f"${r.extracted_requested_amount:,.0f}" if r.extracted_requested_amount else "N/A"
    inc = f"${r.extracted_monthly_income:,.0f}" if r.extracted_monthly_income else "MISSING"
    emi = f"${r.computed_monthly_emi:,.2f}" if r.computed_monthly_emi else "N/A"
    ratio = f"{r.emi_to_income_ratio:.1%}" if r.emi_to_income_ratio is not None else "N/A"
    cibil = str(r.bureau_profile.cibil_score) if r.bureau_profile else "N/A"
    dec = r.decision.value if r.decision else "PENDING"
    print(f"{app_n:<16} | {req:<12} | {inc:<12} | {emi:<13} | {ratio:<8} | {cibil:<6} | {dec:<10}")
print("=" * 105)


Applicant        | Req Loan     | Income       | Monthly EMI   | EMI/Inc  | CIBIL  | Decision  
Alice Sharma     | $500,000     | $125,000     | $16,251.22    | 13.0%    | 785    | SANCTION  
Bob Mehta        | $1,200,000   | $35,000      | $55,651.25    | 159.0%   | 740    | DECLINE   
Charlie Verma    | N/A          | MISSING      | N/A           | N/A      | N/A    | REFER     
David Khan       | $200,000     | $95,000      | $9,275.21     | N/A      | 590    | DECLINE   
